# CelebA-HQ — Stage 5 metadata-only loader/split binding

Run with **Accelerator = None**. Attach the DIRE input Dataset, the code Dataset version containing `dire_modal/`, and the Stage 3 v2 split artifact (or keep its JSON under `/kaggle/working`). This cell validates split IDs, paths, identities, and counts; it does not compare per-run metadata fingerprints, open image contents, create tensors, use GPU, or train. The complete metadata scan can take time; the latest loader package prints progress during path indexing. The attached code package must include the fingerprint-disabled loader; older package versions are detected and rejected with an update instruction.

The output report is written only to `/kaggle/working/celebahq-data-binding-report.json`; the mounted input remains read-only. Do not index/iterate the returned datasets in this stage.

In [ ]:
from pathlib import Path
import inspect
import sys

INPUT_ROOT = Path('/kaggle/input')
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
SPLIT_ARTIFACT = Path('/kaggle/working/celebahq-validation-split-v2-81dcda99aeb9.json')

# Kaggle mounts attached Datasets but does not add their Python packages to sys.path.
# Search only mount roots (and optional src/), never recurse through the image tree.
mount_roots = []
for mount in INPUT_ROOT.iterdir():
    if mount.name == 'datasets' and mount.is_dir():
        for owner in mount.iterdir():
            if owner.is_dir():
                mount_roots.extend(dataset for dataset in owner.iterdir() if dataset.is_dir())
    elif mount.is_dir():
        mount_roots.append(mount)
code_roots = [
    root for mount in mount_roots for root in (mount, mount / 'src')
    if (root / 'dire_modal' / '__init__.py').is_file()
    and (root / 'dire_modal' / 'kaggle_celebahq_data.py').is_file()
]
if len(code_roots) != 1:
    listing = {str(mount): [child.name for child in mount.iterdir()][:12] for mount in mount_roots}
    raise FileNotFoundError(f'Expected one attached code package containing dire_modal; found {code_roots}. Mount listing: {listing}')
CODE_DATASET_ROOT = code_roots[0]
if not DIRE_ROOT.is_dir():
    raise FileNotFoundError(f'Missing DIRE input root: {DIRE_ROOT}')
if not SPLIT_ARTIFACT.is_file():
    raise FileNotFoundError(f'Missing Stage 3 v2 split artifact: {SPLIT_ARTIFACT}')

sys.path.insert(0, str(CODE_DATASET_ROOT))
from dire_modal.kaggle_celebahq_data import build_celebahq_datasets

BUILDER_SOURCE_PATH = Path(inspect.getsourcefile(build_celebahq_datasets)).resolve()
if CODE_DATASET_ROOT.resolve() not in BUILDER_SOURCE_PATH.parents:
    raise RuntimeError(
        f'Imported loader came from {BUILDER_SOURCE_PATH}, outside the selected code Dataset '
        f'{CODE_DATASET_ROOT}. Restart the kernel and attach only the intended code Dataset.'
    )
builder_source = inspect.getsource(build_celebahq_datasets)
if '_source_fingerprint(' in builder_source or 'metadata fingerprint differs' in builder_source:
    raise RuntimeError(
        'The attached Kaggle code Dataset is outdated: its loader still compares the per-mount '
        'metadata fingerprint. Publish/update the code-only dire-modal Dataset with the current '
        'src/dire_modal/kaggle_celebahq_data.py, attach that version, then restart the kernel. '
        f'Loaded file: {BUILDER_SOURCE_PATH}'
    )

print('Code package root added to sys.path:', CODE_DATASET_ROOT)
print('Active binding loader:', BUILDER_SOURCE_PATH)
print('DIRE root:', DIRE_ROOT)
print('Split artifact:', SPLIT_ARTIFACT)

In [ ]:
from collections import Counter
import inspect
import json

def progress(message: str) -> None:
    print('[binding]', message, flush=True)

builder_kwargs = {}
if 'progress_callback' in inspect.signature(build_celebahq_datasets).parameters:
    builder_kwargs['progress_callback'] = progress
else:
    print('Attached code package lacks progress callback; scan still runs without intermediate progress.')

bundle = build_celebahq_datasets(DIRE_ROOT, SPLIT_ARTIFACT, **builder_kwargs)

train_counts = Counter(record.label for record in bundle.train.records)
validation_counts = Counter(record.label for record in bundle.validation.records)
test_counts = {name: len(dataset) for name, dataset in bundle.test_by_generator.items()}

expected_train = {0: 25_200, 1: 36_000}
expected_validation = {0: 2_800, 1: 4_000}
# Inventory counts previously reported for this mounted CelebA-HQ test tree.
expected_test = {
    'real': 1_000, 'dalle2': 500, 'if': 1_000,
    'midjourney': 100, 'stargan': 1_000, 'sdv2': 1_000,
}

assert dict(train_counts) == expected_train, (dict(train_counts), expected_train)
assert dict(validation_counts) == expected_validation, (dict(validation_counts), expected_validation)
assert test_counts == expected_test, (test_counts, expected_test)
assert bundle.validation_id_list_sha256 == (
    'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
)

report = {
    'status': 'pass',
    'protocol': 'celebahq-content-disjoint-exact-count-hash-rank-v2',
    'metadata_fingerprint_check': 'not_performed',
    'validation_id_list_sha256': bundle.validation_id_list_sha256,
    'train_count_by_label': {str(k): v for k, v in sorted(train_counts.items())},
    'validation_count_by_label': {str(k): v for k, v in sorted(validation_counts.items())},
    'test_count_by_generator': test_counts,
    'image_contents_read': False,
    'input_write_attempted': False,
    'gpu_workload_started': False,
    'training_started': False,
}
report_path = Path('/kaggle/working/celebahq-data-binding-report.json')
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved report:', report_path)

## Stage 6 — ImageNet checkpoint identity (read-only, CPU only)

Run only after the Stage 5 binding report says `pass`. This cell locates the cached `resnet50-19c8e357.pth` (or accepts `RESNET50_WEIGHTS_PATH`), reads its bytes once to record file size and full SHA-256, and checks the filename's 8-hex digest prefix. It does not deserialize the checkpoint, create a model, use a GPU, read dataset images, train, or write to mounted inputs. The report is saved under `/kaggle/working`. This is not training approval; seed/budget/retention still need review.


In [ ]:
from pathlib import Path
import hashlib
import json
import os

BINDING_REPORT_PATH = Path('/kaggle/working/celebahq-data-binding-report.json')
CHECKPOINT_REPORT_PATH = Path('/kaggle/working/celebahq-pretrained-checkpoint-report.json')
CHECKPOINT_BASENAME = 'resnet50-19c8e357.pth'
EXPECTED_SHA256_PREFIX = '19c8e357'

if not BINDING_REPORT_PATH.is_file():
    raise FileNotFoundError(f'Run Stage 5 binding first; report missing: {BINDING_REPORT_PATH}')
binding_report = json.loads(BINDING_REPORT_PATH.read_text(encoding='utf-8'))
if binding_report.get('status') != 'pass':
    raise RuntimeError(f'Stage 5 binding did not pass: {binding_report.get("status")!r}')
if binding_report.get('validation_id_list_sha256') != 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd':
    raise RuntimeError('Stage 5 report does not contain the locked validation ID-list hash.')

# Prefer an explicit variable/env override. Otherwise inspect only conventional
# PyTorch cache locations; do not recurse through Kaggle input datasets.
explicit_checkpoint = globals().get('RESNET50_WEIGHTS_PATH') or os.environ.get('RESNET50_WEIGHTS_PATH')
if explicit_checkpoint:
    candidates = [Path(explicit_checkpoint).expanduser()]
else:
    xdg_cache = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')).expanduser()
    torch_home = Path(os.environ.get('TORCH_HOME', xdg_cache / 'torch')).expanduser()
    candidates = [
        torch_home / 'hub' / 'checkpoints' / CHECKPOINT_BASENAME,
        Path.home() / '.cache' / 'torch' / 'hub' / 'checkpoints' / CHECKPOINT_BASENAME,
        Path('/kaggle/working/checkpoints') / CHECKPOINT_BASENAME,
    ]

found = sorted({path.resolve() for path in candidates if path.is_file()})
if not found:
    raise FileNotFoundError(
        f'Could not find {CHECKPOINT_BASENAME} in the standard cache locations. '
        'Set RESNET50_WEIGHTS_PATH to the exact existing checkpoint file and rerun this cell.'
    )
if len(found) != 1:
    raise RuntimeError(
        'Found multiple candidate ImageNet checkpoints; set RESNET50_WEIGHTS_PATH explicitly: '
        + ', '.join(map(str, found))
    )
checkpoint_path = found[0]
if checkpoint_path.name != CHECKPOINT_BASENAME:
    raise ValueError(f'Expected checkpoint basename {CHECKPOINT_BASENAME!r}, got {checkpoint_path.name!r}')

before = checkpoint_path.stat()
digest = hashlib.sha256()
with checkpoint_path.open('rb') as stream:
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
        digest.update(block)
after = checkpoint_path.stat()
stable_during_read = (before.st_size, before.st_mtime_ns, before.st_ino) == (
    after.st_size, after.st_mtime_ns, after.st_ino
)
sha256 = digest.hexdigest()
prefix_matches = sha256.startswith(EXPECTED_SHA256_PREFIX)
status = 'pass' if stable_during_read and prefix_matches else 'fail'
report = {
    'status': status,
    'checkpoint_path': str(checkpoint_path),
    'file_size_bytes': after.st_size,
    'sha256': sha256,
    'filename_sha256_prefix': EXPECTED_SHA256_PREFIX,
    'sha256_prefix_matches_filename': prefix_matches,
    'stable_during_read': stable_during_read,
    'binding_report': str(BINDING_REPORT_PATH),
    'checkpoint_deserialized': False,
    'dataset_image_contents_read': False,
    'gpu_workload_started': False,
    'training_started': False,
    'mounted_input_write_attempted': False,
    'note': 'Full SHA-256 recorded; filename prefix is only an 8-hex reference, not a full independent reference-hash verification.',
}
CHECKPOINT_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved checkpoint report:', CHECKPOINT_REPORT_PATH)
if status != 'pass':
    raise RuntimeError(f'Checkpoint verification failed; inspect {CHECKPOINT_REPORT_PATH}')


## Stage 7 — Training plan + fresh disk preflight (no training)

Run this after Stage 5 and Stage 6 pass, using a code Dataset whose `build_training_plan()` supports the approved parallel schedule. Stage 7 explicitly requires wave 1 seeds 3407 and 3408 concurrently, one T4 per seed, then wave 2 seed 3409. It fails closed if the attached trainer is stale or omits this schedule. It validates the in-memory bundle and current `/kaggle/working` reserve. It does not import PyTorch, query/use a GPU, read image bytes, deserialize weights, or call the trainer. The report is written only under `/kaggle/working`; Stage 7 itself does not start training.


In [ ]:
from pathlib import Path
import inspect
import json
import shutil

PLAN_REPORT_PATH = Path('/kaggle/working/celebahq-training-plan.json')
BINDING_REPORT_PATH = Path('/kaggle/working/celebahq-data-binding-report.json')
CHECKPOINT_REPORT_PATH = Path('/kaggle/working/celebahq-pretrained-checkpoint-report.json')
EXPECTED_VALIDATION_SHA256 = 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
EXPECTED_CHECKPOINT_SHA256 = '19c8e3572231adff6824a2da93fd67b5986919a2e65f8b6007eab4edee220097'

if 'bundle' not in globals() or 'CODE_DATASET_ROOT' not in globals():
    raise RuntimeError('Run Stage 5 setup and binding cells first in this kernel.')
for required_report in (BINDING_REPORT_PATH, CHECKPOINT_REPORT_PATH):
    if not required_report.is_file():
        raise FileNotFoundError(f'Run the required prior stage first; report missing: {required_report}')
binding_report = json.loads(BINDING_REPORT_PATH.read_text(encoding='utf-8'))
checkpoint_report = json.loads(CHECKPOINT_REPORT_PATH.read_text(encoding='utf-8'))
if binding_report.get('status') != 'pass':
    raise RuntimeError('Stage 5 binding report is not pass.')
if binding_report.get('validation_id_list_sha256') != EXPECTED_VALIDATION_SHA256:
    raise RuntimeError('Stage 5 report has the wrong locked validation ID-list hash.')
if checkpoint_report.get('status') != 'pass':
    raise RuntimeError('Stage 6 checkpoint report is not pass.')
if checkpoint_report.get('sha256') != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError('Stage 6 checkpoint digest does not match the recorded full SHA-256.')
if checkpoint_report.get('checkpoint_deserialized') is not False:
    raise RuntimeError('Stage 6 report indicates the checkpoint was deserialized.')
checkpoint_path = Path(checkpoint_report.get('checkpoint_path', '')).expanduser()
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'ImageNet weights file is no longer present: {checkpoint_path}')
if checkpoint_path.stat().st_size != checkpoint_report.get('file_size_bytes'):
    raise RuntimeError('ImageNet checkpoint size changed since Stage 6 audit.')

try:
    from dire_modal.kaggle_celebahq_training import (
        CelebAHQTrainingConfig,
        build_training_plan,
    )
except ImportError as exc:
    raise RuntimeError(
        'The attached code Dataset lacks kaggle_celebahq_training.py. Publish/attach the updated '
        'code-only dire-modal Dataset, restart the kernel, and rerun Stages 5 and 6.'
    ) from exc
training_module_path = Path(inspect.getsourcefile(build_training_plan)).resolve()
if CODE_DATASET_ROOT.resolve() not in training_module_path.parents:
    raise RuntimeError(
        f'Training module came from outside the selected code Dataset: {training_module_path}'
    )

config = CelebAHQTrainingConfig()
plan = build_training_plan(
    bundle,
    config=config,
    weights_report_path=CHECKPOINT_REPORT_PATH,
)
expected_execution_schedule = {
    'mode': 'parallel_independent_seeds',
    'gpu_count': 2,
    'waves': [[3407, 3408], [3409]],
    'per_seed_gpu_visibility': 1,
    'effective_batch_per_seed': 64,
}
if plan.get('execution_schedule') != expected_execution_schedule:
    raise RuntimeError(
        'The attached trainer is stale or has the wrong execution schedule. '
        'Publish/attach the updated dire-modal code Dataset and restart/rebind before Stage 7.'
    )
working = Path('/kaggle/working')
free_bytes = shutil.disk_usage(working).free
suite_reserve_bytes = config.output_reserve_bytes * len(config.seeds)
status = 'ready_for_review' if free_bytes >= suite_reserve_bytes else 'needs_attention'
report = {
    **plan,
    'status': status,
    'execution_schedule': expected_execution_schedule,
    'training_module': str(training_module_path),
    'checkpoint': {
        'path': str(checkpoint_path.resolve()),
        'file_size_bytes': checkpoint_report['file_size_bytes'],
        'sha256': checkpoint_report['sha256'],
    },
    'disk_preflight': {
        'working_path': str(working.resolve()),
        'free_bytes': free_bytes,
        'reserve_per_seed_bytes': config.output_reserve_bytes,
        'seeds': len(config.seeds),
        'suite_reserve_bytes': suite_reserve_bytes,
        'passes': free_bytes >= suite_reserve_bytes,
    },
    'image_contents_read': False,
    'checkpoint_deserialized': False,
    'gpu_metadata_queried': False,
    'gpu_workload_started': False,
    'training_started': False,
    'mounted_input_write_attempted': False,
    'next_gate': 'review plan; do not call the training entry point without a separate explicit go-ahead',
}
PLAN_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved plan and disk report:', PLAN_REPORT_PATH)


## Stage 8 — Two-T4 metadata-only preflight

Run after Stage 7, with Kaggle Accelerator set to **GPU T4 x2**. This freshly queries device names/UUIDs/memory using `nvidia-smi`, requires exactly two distinct Tesla T4 devices, and writes a report under `/kaggle/working`. It does not import PyTorch, allocate tensors, start a GPU workload, read images/checkpoints, or train. A pass only confirms the two-device metadata gate; it is not authorization to start training.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess

PLAN_REPORT_PATH = Path('/kaggle/working/celebahq-training-plan.json')
GPU_REPORT_PATH = Path('/kaggle/working/celebahq-gpu-metadata-report.json')
if not PLAN_REPORT_PATH.is_file():
    raise FileNotFoundError(f'Run Stage 7 first: {PLAN_REPORT_PATH}')
plan = json.loads(PLAN_REPORT_PATH.read_text(encoding='utf-8'))
if plan.get('status') != 'ready_for_review' or plan.get('training_started') is not False:
    raise RuntimeError('Stage 7 plan is not in the expected no-training review state.')
command = [
    'nvidia-smi',
    '--query-gpu=name,uuid,memory.total,memory.free,driver_version',
    '--format=csv,noheader,nounits',
]
try:
    result = subprocess.run(command, capture_output=True, text=True, timeout=15, check=False)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or f'nvidia-smi exited {result.returncode}')
    devices = []
    for row in csv.reader(result.stdout.splitlines(), skipinitialspace=True):
        values = [value.strip() for value in row]
        if not values or not any(values):
            continue
        if len(values) != 5:
            raise RuntimeError(f'unexpected nvidia-smi row: {values}')
        devices.append({
            'name': values[0], 'uuid': values[1],
            'memory_total_mib': values[2], 'memory_free_mib': values[3],
            'driver_version': values[4],
        })
    valid = (
        len(devices) == 2
        and all('T4' in device['name'] and device['uuid'] for device in devices)
        and len({device['uuid'] for device in devices}) == 2
    )
    report = {
        'status': 'pass' if valid else 'needs_attention',
        'devices': devices,
        'expected': 'exactly two distinct NVIDIA Tesla T4 devices',
        'gpu_metadata_queried': True,
        'gpu_workload_started': False,
        'training_started': False,
        'image_contents_read': False,
        'checkpoint_deserialized': False,
        'mounted_input_write_attempted': False,
        'note': 'Metadata-only gate; does not authorize training.',
    }
except Exception as exc:
    report = {
        'status': 'needs_attention',
        'error': f'{type(exc).__name__}: {exc}',
        'gpu_metadata_queried': True,
        'gpu_workload_started': False,
        'training_started': False,
        'image_contents_read': False,
        'checkpoint_deserialized': False,
        'mounted_input_write_attempted': False,
        'note': 'No training was started; verify Kaggle Accelerator/runtime.',
    }
GPU_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved metadata-only GPU report:', GPU_REPORT_PATH)


## Stage 9 — Launch the approved two-T4 seed schedule

**This cell starts GPU training.** The user has explicitly approved beginning this run. Run only after Stage 7 produced a fresh plan containing `parallel_independent_seeds` and Stage 8 passed with two distinct T4 UUIDs in this same Kaggle runtime. It rechecks reports, attached code source, bundle availability and current free space; the trainer independently repeats its gates. Before GPU workers start, the runner performs one CPU-only legacy conversion with `weights_only=False` on the user-confirmed official checkpoint, verifies strict ResNet-50 keys, writes a weights-only-compatible copy, and confirms it reloads with `weights_only=True`. Each worker then loads only that converted safe copy. The schedule runs seeds 3407/3408 concurrently, each pinned to one T4, then seed 3409 after both finish. Existing per-seed checkpoints resume when compatible. Outputs and a final ledger are written only under `/kaggle/working`. Do not interrupt unless necessary; if interrupted, inspect per-seed `run.json` and resume with the same Stage 7/8 gates.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import inspect
import json
import shutil

PLAN_REPORT_PATH = Path('/kaggle/working/celebahq-training-plan.json')
BINDING_REPORT_PATH = Path('/kaggle/working/celebahq-data-binding-report.json')
CHECKPOINT_REPORT_PATH = Path('/kaggle/working/celebahq-pretrained-checkpoint-report.json')
GPU_REPORT_PATH = Path('/kaggle/working/celebahq-gpu-metadata-report.json')
RUN_ROOT = Path('/kaggle/working/celebahq-run-v2')
RUN_REPORT_PATH = Path('/kaggle/working/celebahq-parallel-run-report.json')
EXPECTED_VALIDATION_SHA256 = 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
EXPECTED_CHECKPOINT_SHA256 = '19c8e3572231adff6824a2da93fd67b5986919a2e65f8b6007eab4edee220097'

required_globals = ('bundle', 'CODE_DATASET_ROOT', 'DIRE_ROOT', 'SPLIT_ARTIFACT')
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f'Run Stage 5 setup/binding first in this kernel; missing {missing_globals}')
for path in (PLAN_REPORT_PATH, BINDING_REPORT_PATH, CHECKPOINT_REPORT_PATH, GPU_REPORT_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Required prior-stage report missing: {path}')
plan = json.loads(PLAN_REPORT_PATH.read_text(encoding='utf-8'))
binding_report = json.loads(BINDING_REPORT_PATH.read_text(encoding='utf-8'))
checkpoint_report = json.loads(CHECKPOINT_REPORT_PATH.read_text(encoding='utf-8'))
gpu_report = json.loads(GPU_REPORT_PATH.read_text(encoding='utf-8'))
if plan.get('status') != 'ready_for_review' or plan.get('training_started') is not False:
    raise RuntimeError('Stage 7 plan is not a clean ready_for_review plan.')
if plan.get('inventory', {}).get('validation_id_list_sha256') != EXPECTED_VALIDATION_SHA256:
    raise RuntimeError('Plan validation-ID hash differs from the approved split.')
if plan.get('execution_schedule') != {
    'mode': 'parallel_independent_seeds',
    'gpu_count': 2,
    'waves': [[3407, 3408], [3409]],
    'per_seed_gpu_visibility': 1,
    'effective_batch_per_seed': 64,
}:
    raise RuntimeError('Fresh Stage 7 plan lacks the approved parallel-seed schedule.')
if binding_report.get('status') != 'pass' or binding_report.get('training_started') is not False:
    raise RuntimeError('Stage 5 report did not pass or indicates training was started.')
if checkpoint_report.get('status') != 'pass' or checkpoint_report.get('sha256') != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError('Stage 6 checkpoint audit did not pass the expected digest.')
if checkpoint_report.get('checkpoint_deserialized') is not False:
    raise RuntimeError('Stage 6 report indicates checkpoint deserialization.')
if gpu_report.get('status') != 'pass' or gpu_report.get('training_started') is not False:
    raise RuntimeError('Stage 8 two-T4 metadata gate did not pass.')
if gpu_report.get('gpu_workload_started') is not False:
    raise RuntimeError('Stage 8 report indicates a GPU workload was started.')
if len(gpu_report.get('devices', [])) != 2 or any('T4' not in d.get('name', '') for d in gpu_report['devices']):
    raise RuntimeError('Stage 8 report does not contain exactly two T4 devices.')
if len({d.get('uuid') for d in gpu_report['devices']}) != 2:
    raise RuntimeError('Stage 8 T4 UUIDs are missing or duplicated.')

from dire_modal.kaggle_celebahq_training import (
    CelebAHQTrainingConfig,
    run_celebahq_parallel_seed_schedule,
)
training_module_path = Path(inspect.getsourcefile(run_celebahq_parallel_seed_schedule)).resolve()
if CODE_DATASET_ROOT.resolve() not in training_module_path.parents:
    raise RuntimeError(f'Trainer did not load from the selected Kaggle code Dataset: {training_module_path}')
config = CelebAHQTrainingConfig()
free_bytes = shutil.disk_usage('/kaggle/working').free
required_bytes = config.output_reserve_bytes * len(config.seeds)
if free_bytes < required_bytes:
    raise OSError(f'Fresh free space {free_bytes} is below the {required_bytes}-byte full-suite reserve.')

print('All Stage 9 gates pass. Launching the two-GPU independent-seed schedule now.', flush=True)
try:
    worker_results = run_celebahq_parallel_seed_schedule(
        dire_root=DIRE_ROOT,
        split_artifact_path=SPLIT_ARTIFACT,
        checkpoint_report_path=CHECKPOINT_REPORT_PATH,
        plan_report_path=PLAN_REPORT_PATH,
        gpu_report_path=GPU_REPORT_PATH,
        output_root=RUN_ROOT,
        config=config,
        resume=True,
    )
    seed_runs = {}
    for seed in config.seeds:
        run_json = RUN_ROOT / f'seed-{seed}' / 'run.json'
        if not run_json.is_file():
            raise FileNotFoundError(f'Worker exited successfully but run ledger is missing: {run_json}')
        seed_runs[str(seed)] = json.loads(run_json.read_text(encoding='utf-8'))
    conversion_report_path = RUN_ROOT / 'pretrained' / 'resnet50-19c8e357-weights-only.pt.json'
    if not conversion_report_path.is_file():
        raise FileNotFoundError(f'One-time safe checkpoint conversion report is missing: {conversion_report_path}')
    conversion_report = json.loads(conversion_report_path.read_text(encoding='utf-8'))
    if conversion_report.get('status') != 'pass' or conversion_report.get('legacy_deserialization_count') != 1:
        raise RuntimeError('Trusted checkpoint conversion did not verify exactly one legacy deserialization.')
    run_report = {
        'status': 'complete',
        'execution_schedule': plan['execution_schedule'],
        'worker_results': worker_results,
        'seed_runs': seed_runs,
        'checkpoint_conversion_report': conversion_report,
        'training_started': True,
        'gpu_workload_started': True,
        'test_data_used_for_selection': False,
        'mounted_input_write_attempted': False,
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    RUN_REPORT_PATH.write_text(json.dumps(run_report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(run_report, indent=2))
except Exception as exc:
    failure_report = {
        'status': 'failed_or_interrupted',
        'error': f'{type(exc).__name__}: {exc}',
        'training_launch_attempted': True,
        'inspect_seed_run_json_and_last_resume_before_retry': True,
        'mounted_input_write_attempted': False,
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    RUN_REPORT_PATH.write_text(json.dumps(failure_report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(failure_report, indent=2))
    raise
print('Saved Stage 9 run report:', RUN_REPORT_PATH)


## Stage 10 — Held-out generator evaluation + paired bootstrap

Run only after Stage 9 reports `complete` for all three seeds. This cell uses each seed's validation-selected `best_model` exactly once, performs inference on the untouched test sets, and never changes/selects a checkpoint using test metrics. It reports raw-prevalence AP, 50:50-prior weighted AP, AUROC, ACC@0.5, counts, and paired stratified 95% percentile-bootstrap CIs. Real test IDs are resampled jointly across generator families; fake IDs are independently resampled within each family. `dalle2`, `if`, `midjourney`, and `stargan` define the primary macro-AP; `sdv2` remains a seen-generator control. Per-seed CIs and seed-to-seed SD are reported separately. Bootstrap uses 2,000 replicates with fixed seed 3407. Test images are decoded here; outputs are written under `/kaggle/working/celebahq-evaluation-v1` only. No training occurs.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import statistics

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import resnet50

from dire_modal.kaggle_celebahq_data import CelebAHQDataset
from dire_modal.kaggle_celebahq_training import CelebAHQTrainingConfig, validate_bundle
from dire_modal.kaggle_celebahq_evaluation import (
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_SEED,
    PRIMARY_HOLDOUT_FAMILIES,
    paired_stratified_bootstrap,
)

RUN_REPORT_PATH = Path('/kaggle/working/celebahq-parallel-run-report.json')
EVAL_ROOT = Path('/kaggle/working/celebahq-evaluation-v1')
EVAL_REPORT_PATH = EVAL_ROOT / 'celebahq-stage10-evaluation.json'
PREDICTIONS_PATH = EVAL_ROOT / 'celebahq-stage10-predictions.csv'
SEEDS = (3407, 3408, 3409)
GENERATOR_FAMILIES = ('dalle2', 'if', 'midjourney', 'stargan', 'sdv2')
EXPECTED_TEST_COUNTS = {
    'real': 1000, 'dalle2': 500, 'if': 1000,
    'midjourney': 100, 'stargan': 1000, 'sdv2': 1000,
}

if 'bundle' not in globals() or 'CODE_DATASET_ROOT' not in globals():
    raise RuntimeError('Stage 10 needs the Stage 5 bundle in this kernel; run setup and binding cells first.')
if not RUN_REPORT_PATH.is_file():
    raise FileNotFoundError(f'Stage 9 run report not found: {RUN_REPORT_PATH}')
run_report = json.loads(RUN_REPORT_PATH.read_text(encoding='utf-8'))
if run_report.get('status') != 'complete' or run_report.get('training_started') is not True:
    raise RuntimeError('Stage 9 did not complete successfully; evaluation is blocked.')
if run_report.get('test_data_used_for_selection') is not False:
    raise RuntimeError('Stage 9 report indicates test data was used for selection.')
conversion_report = run_report.get('checkpoint_conversion_report', {})
if (
    conversion_report.get('status') != 'pass'
    or conversion_report.get('legacy_deserialization_count') != 1
    or conversion_report.get('safe_checkpoint_verified_with_weights_only') is not True
):
    raise RuntimeError('Stage 9 did not record a passing one-time trusted checkpoint conversion.')
if set(run_report.get('seed_runs', {})) != {str(seed) for seed in SEEDS}:
    raise RuntimeError('Stage 9 report does not contain exactly the three approved seeds.')
for seed in SEEDS:
    seed_run = run_report['seed_runs'][str(seed)]
    if seed_run.get('status') not in ('complete', 'early_stopped'):
        raise RuntimeError(f'Seed {seed} did not finish: {seed_run.get("status")!r}')
    if seed_run.get('test_data_used_for_selection') is not False:
        raise RuntimeError(f'Seed {seed} report indicates test data was used for selection.')

inventory = validate_bundle(bundle)
if inventory['validation_id_list_sha256'] != 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd':
    raise RuntimeError('Current bundle validation-ID hash differs from the approved split.')
if inventory['test_count_by_generator'] != EXPECTED_TEST_COUNTS:
    raise RuntimeError(f'Current test inventory differs from the locked counts: {inventory["test_count_by_generator"]}')
if not torch.cuda.is_available():
    raise RuntimeError('Stage 10 evaluation requires the Kaggle GPU runtime used for Stage 9.')
device = torch.device('cuda:0')
config = CelebAHQTrainingConfig()
interpolation = transforms.InterpolationMode.BILINEAR
eval_transform = transforms.Compose([
    transforms.Resize(config.resize_short_edge, interpolation=interpolation, antialias=True),
    transforms.CenterCrop(config.crop_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


def score_dataset(model, dataset, expected_label):
    records = tuple(sorted(dataset.records, key=lambda record: record.relative_path))
    if any(int(record.label) != expected_label for record in records):
        raise ValueError(f'Unexpected label in group expected to contain label {expected_label}.')
    eval_dataset = CelebAHQDataset(dataset.root, records, transform=eval_transform)
    loader = DataLoader(
        eval_dataset, batch_size=config.microbatch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True, persistent_workers=False,
    )
    ids = [str(record.relative_path) for record in records]
    scores = []
    with torch.inference_mode():
        for images, labels in loader:
            if not torch.all(labels == expected_label):
                raise ValueError('DataLoader labels disagree with locked test group labels.')
            logits = model(images.to(device, non_blocking=True)).reshape(-1)
            scores.extend(float(value) for value in torch.sigmoid(logits).cpu().tolist())
    if len(scores) != len(ids):
        raise RuntimeError(f'Inference count mismatch: got {len(scores)} scores for {len(ids)} IDs.')
    return ids, scores


EVAL_ROOT.mkdir(parents=True, exist_ok=True)
seed_results = {}
all_predictions = []
for seed in SEEDS:
    seed_run = run_report['seed_runs'][str(seed)]
    best_path = Path(seed_run['best_model_checkpoint'])
    if not best_path.is_file():
        raise FileNotFoundError(f'Seed {seed} best checkpoint is missing: {best_path}')
    best_sha = hashlib.sha256(best_path.read_bytes()).hexdigest()
    best_state = torch.load(best_path, map_location='cpu', weights_only=True)
    if not isinstance(best_state, dict) or best_state.get('seed') != seed:
        raise ValueError(f'Seed {seed} best checkpoint metadata is invalid.')
    if int(best_state.get('epoch', -1)) != int(seed_run['best_epoch']):
        raise ValueError(f'Seed {seed} best checkpoint epoch differs from Stage 9 ledger.')
    if best_state.get('validation_id_list_sha256') != inventory['validation_id_list_sha256']:
        raise ValueError(f'Seed {seed} best checkpoint validation IDs differ from current bundle.')
    if best_state.get('config_sha256') != seed_run.get('config_sha256'):
        raise ValueError(f'Seed {seed} best checkpoint config hash differs from Stage 9 ledger.')

    model = resnet50(weights=None)
    model.fc = torch.nn.Linear(2048, 1)
    model.load_state_dict(best_state['model_state'], strict=True)
    model.to(device)
    model.eval()
    del best_state

    real_ids, real_scores = score_dataset(model, bundle.test_by_generator['real'], 0)
    fake_scores_by_family = {}
    fake_ids_by_family = {}
    for family in GENERATOR_FAMILIES:
        fake_ids, fake_scores = score_dataset(model, bundle.test_by_generator[family], 1)
        fake_ids_by_family[family] = fake_ids
        fake_scores_by_family[family] = fake_scores
    bootstrap = paired_stratified_bootstrap(
        real_ids=real_ids,
        real_scores=real_scores,
        fake_scores_by_family=fake_scores_by_family,
        families=PRIMARY_HOLDOUT_FAMILIES,
        replicates=BOOTSTRAP_REPLICATES,
        seed=BOOTSTRAP_SEED,
    )
    seed_results[str(seed)] = {
        'status': seed_run['status'],
        'best_epoch': seed_run['best_epoch'],
        'best_validation_ap': seed_run['best_validation_ap'],
        'best_model_checkpoint': str(best_path),
        'best_model_sha256': best_sha,
        'primary_macro_ap_balanced_prior_50_50': bootstrap['point_macro_ap_balanced_prior_50_50'],
        'primary_macro_ap_ci95': bootstrap['macro_ap_balanced_prior_50_50_ci95'],
        'metrics_by_generator': bootstrap['point_by_family'],
        'ci95_by_generator': bootstrap['ci95_by_family'],
        'bootstrap': {
            'replicates': bootstrap['replicates'],
            'seed': bootstrap['seed'],
            'ci_method': bootstrap['ci_method'],
            'pairing': bootstrap['pairing'],
        },
    }
    for sample_id, score in zip(real_ids, real_scores):
        all_predictions.append((seed, 'real', sample_id, 0, score))
    for family in GENERATOR_FAMILIES:
        for sample_id, score in zip(fake_ids_by_family[family], fake_scores_by_family[family]):
            all_predictions.append((seed, family, sample_id, 1, score))
    del model, real_scores, fake_scores_by_family
    torch.cuda.empty_cache()

seed_macro_values = [
    float(seed_results[str(seed)]['primary_macro_ap_balanced_prior_50_50'])
    for seed in SEEDS
]
seed_metric_variability = {}
for family in GENERATOR_FAMILIES:
    seed_metric_variability[family] = {}
    for metric in ('ap_raw_prevalence', 'ap_balanced_prior_50_50', 'auroc', 'acc_at_0_5'):
        values = [float(seed_results[str(seed)]['metrics_by_generator'][family][metric]) for seed in SEEDS]
        seed_metric_variability[family][metric] = {
            'mean': statistics.mean(values),
            'population_sd': statistics.pstdev(values),
            'per_seed': {str(seed): value for seed, value in zip(SEEDS, values)},
        }
seed_variability = {
    'primary_macro_ap_balanced_prior_50_50_mean': statistics.mean(seed_macro_values),
    'primary_macro_ap_balanced_prior_50_50_population_sd': statistics.pstdev(seed_macro_values),
    'primary_macro_ap_balanced_prior_50_50_per_seed': {
        str(seed): seed_results[str(seed)]['primary_macro_ap_balanced_prior_50_50'] for seed in SEEDS
    },
    'per_generator_metrics': seed_metric_variability,
    'note': 'Seed variability is reported separately; bootstrap CIs are per seed, not pooled across seeds.',
}
report = {
    'status': 'pass',
    'stage': 10,
    'track': 'CelebA-HQ-SD-v2 research extension',
    'execution_schedule': run_report['execution_schedule'],
    'primary_metric': 'macro average of per-family AP with sample weights imposing 50:50 real/fake prior',
    'primary_holdout_families': list(PRIMARY_HOLDOUT_FAMILIES),
    'seen_generator_control': 'sdv2',
    'label_definition': {'real': 0, 'fake': 1},
    'test_inventory': inventory['test_count_by_generator'],
    'validation_id_list_sha256': inventory['validation_id_list_sha256'],
    'preprocessing': 'Resize(256) -> CenterCrop(224) -> ToTensor -> ImageNet Normalize',
    'threshold': {'acc_at_0_5': 0.5, 'tuned_on_test': False},
    'bootstrap': {
        'replicates': BOOTSTRAP_REPLICATES,
        'seed': BOOTSTRAP_SEED,
        'ci_method': 'paired stratified percentile 95%',
        'real_ids_shared_across_families': True,
        'fake_ids_resampled_independently_by_family': True,
    },
    'seed_results': seed_results,
    'seed_variability': seed_variability,
    'test_data_used_for_checkpoint_selection': False,
    'training_started': False,
    'gpu_workload_started': True,
    'image_contents_read': True,
    'mounted_input_write_attempted': False,
    'predictions_csv': str(PREDICTIONS_PATH),
}
with PREDICTIONS_PATH.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(('seed', 'generator', 'test_id', 'label', 'p_fake'))
    writer.writerows(all_predictions)
EVAL_REPORT_PATH.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved Stage 10 report:', EVAL_REPORT_PATH)
print('Saved test predictions:', PREDICTIONS_PATH)
